# FloorPlanCAD Object Detection
**Multimodal detection model — Mamba + Attention + Heatmap Head**

## Checklist trước khi chạy
1. `Runtime` → `Change runtime type` → **GPU (T4 hoặc A100)**
2. Mount Google Drive (nếu muốn lưu data + checkpoint lâu dài)
3. Chạy từng cell theo thứ tự

## 1. Kiểm tra GPU

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'GPU            : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Mount Google Drive (khuyến nghị — để lưu data & checkpoint)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục làm việc trên Drive
import os
WORK_DIR = '/content/drive/MyDrive/FloorPlanCAD'
os.makedirs(WORK_DIR, exist_ok=True)
print(f'Work dir: {WORK_DIR}')

## 3. Clone repo

In [ ]:
%cd /content
!git clone https://github.com/DaniYLab/object_detection.git
%cd /content/object_detection
!ls

## 4. Cài dependencies

In [ ]:
# Colab đã có PyTorch CUDA — chỉ cần cài thêm thư viện còn lại
!pip install -q gdown Pillow torchvision tqdm

# Verify
import torch, torchvision
print(f'torch     : {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'CUDA      : {torch.version.cuda}')

## 5. Download Dataset từ Google Drive
> ⚠️ **Lưu ý**: Dataset ~5GB. Nếu đã có trên Drive rồi thì skip cell này và symlink thẳng vào.

In [ ]:
# Option A: Download mới (lần đầu)
DATA_DIR = f'{WORK_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# Link dữ liệu của chúng ta đã hardcode trong script
!python scripts/data/download_gdrive.py

# Sau khi download xong, build processed dataset
# (có thể mất 20-40 phút lần đầu)
# !python scripts/data/build_dataset.py

In [ ]:
# Option B: Dataset đã có trên Drive → chỉ tạo symlink vào /content
# Uncomment nếu dùng Option B

# DATASET_ON_DRIVE = f'{WORK_DIR}/data/FloorPlanCAD_dataset'
# !ln -sf {DATASET_ON_DRIVE} /content/object_detection/data/FloorPlanCAD_dataset
# print('Symlink created!')

## 6. Verify dataset

In [ ]:
import sys
sys.path.insert(0, '/content/object_detection')

from src.data.dataset import FloorPlanDataset, CLASS_NAMES, NUM_CLASSES

DATA_ROOT = './data/FloorPlanCAD_dataset'

train_ds = FloorPlanDataset(DATA_ROOT, split='train', image_size=512)
val_ds   = FloorPlanDataset(DATA_ROOT, split='test',  image_size=512)

print(f'Train samples : {len(train_ds):,}')
print(f'Val samples   : {len(val_ds):,}')
print(f'Num classes   : {NUM_CLASSES}')

sample = train_ds[0]
print(f'Image shape   : {sample["image"].shape}')
print(f'Heatmap shape : {sample["heatmap"].shape}')
print(f'Classes in sample: {[CLASS_NAMES[i] for i in sample["class_ids"]]}')

## 7. Verify model

In [ ]:
from src.models.detector import FloorPlanDetector

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = FloorPlanDetector(
    image_size  = 512,
    model_dim   = 512,
    num_classes = NUM_CLASSES,
    num_blocks  = 4,
).to(device)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model params : {n_params:.1f}M')

# Quick forward test
img  = torch.randn(2, 3, 512, 512).to(device)
tids = torch.randint(0, 32000, (2, 16)).to(device)
cids = torch.tensor([3, 10]).to(device)
out  = model(img, tids, cids)
print(f'Output shape : {out.shape}')  # [2, 35, 64, 64]
print('Model OK!')

## 8. Training

### Config gợi ý theo GPU:
| GPU | batch_size | image_size | model_dim | VRAM |
|-----|-----------|------------|-----------|------|
| T4 (16GB)  | 8  | 512 | 512 | ~14GB |
| A100 (40GB)| 16 | 512 | 768 | ~35GB |
| L4 (24GB)  | 12 | 512 | 512 | ~20GB |

In [ ]:
# === CONFIG ===
CONFIG = dict(
    data_root    = './data/FloorPlanCAD_dataset',
    ckpt_dir     = f'{WORK_DIR}/checkpoints',   # lưu trên Drive
    image_size   = 512,
    crop_size    = 128,
    batch_size   = 8,       # Giảm xuống 4 nếu OOM
    num_workers  = 2,
    model_dim    = 512,
    num_blocks   = 4,
    epochs       = 50,
    lr           = 1e-4,
    log_interval = 50,
)
print('Config:', CONFIG)

In [ ]:
# Chạy training qua CLI (có thể theo dõi loss trực tiếp)
!python train.py \
    --data_root    {CONFIG['data_root']} \
    --ckpt_dir     {CONFIG['ckpt_dir']} \
    --image_size   {CONFIG['image_size']} \
    --batch_size   {CONFIG['batch_size']} \
    --num_workers  {CONFIG['num_workers']} \
    --model_dim    {CONFIG['model_dim']} \
    --num_blocks   {CONFIG['num_blocks']} \
    --epochs       {CONFIG['epochs']} \
    --lr           {CONFIG['lr']} \
    --log_interval {CONFIG['log_interval']}

## 9. Load checkpoint & Inference

In [ ]:
import torch
from src.models.detector import FloorPlanDetector
from src.data.dataset import FloorPlanDataset, CLASS_NAMES, NUM_CLASSES

CKPT_PATH = f'{WORK_DIR}/checkpoints/best.pt'

ckpt  = torch.load(CKPT_PATH, map_location='cpu')
model = FloorPlanDetector(image_size=512, model_dim=512, num_classes=NUM_CLASSES, num_blocks=4)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]} | Val IoU: {ckpt["val_iou"]:.4f}')

In [ ]:
# Visualize prediction trên một sample
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

ds     = FloorPlanDataset('./data/FloorPlanCAD_dataset', split='test', image_size=512)
sample = ds[0]

img_t  = sample['image'].unsqueeze(0)        # [1,3,H,W]
tids   = torch.randint(0, 32000, (1, 16))    # placeholder
cids   = torch.tensor([sample['class_ids'][0] if sample['class_ids'] else 0])

with torch.no_grad():
    pred = model(img_t, tids, cids)[0]       # [35, h, w]

# Plot original + top predicted heatmaps
img_np = ((sample['image'].permute(1,2,0).numpy() * 0.5 + 0.5) * 255).astype('uint8')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes[0,0].imshow(img_np); axes[0,0].set_title('Original')

top_classes = pred.max(-1).values.max(-1).values.topk(7).indices
for i, cid in enumerate(top_classes):
    ax = axes[(i+1)//4, (i+1)%4]
    ax.imshow(pred[cid].numpy(), cmap='hot', vmin=0, vmax=1)
    ax.set_title(f'{CLASS_NAMES[cid]}\n(max={pred[cid].max():.2f})')

for ax in axes.flat: ax.axis('off')
plt.tight_layout()
plt.savefig('prediction_viz.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved prediction_viz.png')